# PC.2 — Acesso a dados textuais no spaCy e ao corpora SUBTLEX-pt-BR

**Disciplina:** Processamento de Linguagem Natural
**Exercício:** PC.2 — *"Exemplifique o acesso a dados textuais prontamente disponíveis na biblioteca
SpaCy. Adicionalmente, exemplifique o acesso ao corpora SUBTLEX-pt-br. Exemplifique, através de um
pequeno trecho de código executável, o acesso aos dados e metadados do corpora aplicável a um
problema simples de PLN."*

---

## Organização do notebook

| Parte | Conteúdo |
|---|---|
| 0 | Setup e carregamento do modelo |
| 1 | Dados textuais prontamente disponíveis no spaCy |
| 2 | Acesso ao corpora SUBTLEX-pt-BR (dados + metadados) |
| 3 | Aplicação a um problema simples de PLN |

## Conceitos envolvidos

- **Corpus / corpora**: conjunto finito de dados linguísticos reais de uma língua. O SUBTLEX-pt-BR é
  um corpus *derivado*: não distribui os textos, e sim a **tabela de frequências** extraída deles.
- **Metadados / anotação**: marcações sobre os dados linguísticos que permitem classificá-los ou
  usá-los de forma específica. No SUBTLEX-pt-BR os metadados são `FREQcount`, `CDcount` e
  `Spellcheck`; no spaCy, o bloco `nlp.meta` (corpora de treino, licenças, rótulos, acurácia).
- **Diversidade contextual (CD)**: em quantos documentos distintos (aqui, arquivos de legenda) a
  palavra aparece. Distingue palavra *corrente* de palavra *concentrada em poucos documentos*.
- **Pipeline de PLN**: pré-processamento → extração de características → modelo de linguagem →
  modelo de IA. Este notebook trabalha nas duas primeiras etapas.

## Parte 0 — Setup

In [3]:
# Execute uma única vez, se ainda não instalou as dependências:
# !pip install -r requirements.txt
# !python -m spacy download pt_core_news_sm

In [4]:
import os
import math
import urllib.request

import pandas as pd
import spacy

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

nlp = spacy.load("pt_core_news_sm")
print("spaCy:", spacy.__version__)
print("modelo:", nlp.meta["lang"] + "_" + nlp.meta["name"], nlp.meta["version"])
print("componentes do pipeline:", nlp.pipe_names)

spaCy: 3.8.7
modelo: pt_core_news_sm 3.8.0
componentes do pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


---
## Parte 1 — Dados textuais prontamente disponíveis no spaCy

Uma observação importante antes do código: **o spaCy não é uma biblioteca de corpora como o NLTK**.
O NLTK distribui textos completos (Machado, Mac-Morpho, Floresta Sintá(c)tica, Gutenberg). O spaCy
segue outra filosofia — ele entrega *ferramentas* e *modelos treinados*. O que está "prontamente
disponível" nele se divide em três categorias:

1. **Dados textuais literais embutidos no pacote** — frases de exemplo por idioma, listas de
   *stop words*, exceções de tokenização (abreviaturas, contrações, emoticons), listas de numerais.
2. **Modelos estatísticos pré-treinados** — que carregam um vocabulário/léxico e sabem produzir
   lema, POS, dependência sintática e entidades nomeadas.
3. **Metadados do modelo** (`nlp.meta`) — onde ficam registrados **quais corpora foram usados no
   treinamento**, com autoria e licença. É por aí que o spaCy se conecta a corpora reais.

### 1.1 Frases de exemplo embutidas na biblioteca

In [5]:
from spacy.lang.pt.examples import sentences as frases_pt
from spacy.lang.en.examples import sentences as frases_en

print("Português (spacy.lang.pt.examples):")
for s in frases_pt:
    print("  •", s)

print("\nInglês (spacy.lang.en.examples), 2 primeiras:")
for s in frases_en[:2]:
    print("  •", s)

Português (spacy.lang.pt.examples):
  • Apple está querendo comprar uma startup do Reino Unido por 100 milhões de dólares
  • Carros autônomos empurram a responsabilidade do seguro para os fabricantes.São Francisco considera banir os robôs de entrega que andam pelas calçadas
  • Londres é a maior cidade do Reino Unido

Inglês (spacy.lang.en.examples), 2 primeiras:
  • Apple is looking at buying U.K. startup for $1 billion
  • Autonomous cars shift insurance liability toward manufacturers


### 1.2 Stop words da língua portuguesa

Dado textual embutido, usado em pré-processamento para remoção de palavras de alta frequência e
baixo poder discriminativo.

In [6]:
from spacy.lang.pt.stop_words import STOP_WORDS as STOP_PT

print(f"Total de stop words em português: {len(STOP_PT)}")
print("Amostra:", sorted(STOP_PT)[:20])
print("\n'não' é stop word?", "não" in STOP_PT, "| 'computador' é stop word?", "computador" in STOP_PT)

Total de stop words em português: 416
Amostra: ['a', 'acerca', 'ademais', 'adeus', 'agora', 'ainda', 'algo', 'algumas', 'alguns', 'ali', 'além', 'ambas', 'ambos', 'antes', 'ao', 'aos', 'apenas', 'apoia', 'apoio', 'apontar']

'não' é stop word? True | 'computador' é stop word? False


### 1.3 Exceções de tokenização

Regras casadas à mão que impedem o tokenizador de quebrar abreviaturas em pontos finais — ligado
diretamente ao problema do exercício TC.24 (pontuação não é marcador confiável de fim de sentença).

In [7]:
from spacy.lang.pt.tokenizer_exceptions import TOKENIZER_EXCEPTIONS as EXC_PT

abreviaturas = [k for k in EXC_PT if k.endswith(".") and len(k) > 2]
print(f"Total de exceções de tokenização (pt): {len(EXC_PT)}")
print("Abreviaturas tratadas (amostra):", abreviaturas[:20])

# Efeito prático: o ponto de "Sr." não encerra a sentença
doc = nlp("O Sr. Silva abriu o chamado. O técnico respondeu.")
print("\nTokens:", [t.text for t in doc])
print("Sentenças:", [s.text for s in doc.sents])

Total de exceções de tokenização (pt): 223
Abreviaturas tratadas (amostra): ['._.', '°c.', '°f.', '°k.', '°C.', '°F.', '°K.', 'Adm.', 'Art.', 'art.', 'Av.', 'av.', 'Cia.', 'dom.', 'Dr.', 'dr.', 'e.g.', 'E.g.', 'E.G.', 'ed.']

Tokens: ['O', 'Sr.', 'Silva', 'abriu', 'o', 'chamado', '.', 'O', 'técnico', 'respondeu', '.']
Sentenças: ['O Sr. Silva abriu o chamado.', 'O técnico respondeu.']


### 1.4 Metadados do modelo — a ponte para os corpora reais

`nlp.meta` é o equivalente, no spaCy, aos metadados de um corpus: diz **de onde vieram os dados**,
sob que licença, quais rótulos o modelo conhece e qual sua acurácia medida.

In [8]:
meta = nlp.meta

print("Licença do modelo:", meta["license"])
print("Vetores:", meta["vectors"])
print("\nCorpora de origem (sources) usados no treinamento:")
for src in meta["sources"]:
    print(f"  • {src['name']}")
    print(f"      licença: {src['license']}")
    print(f"      url    : {src['url']}")

print("\nRótulos de entidade nomeada:", meta["labels"]["ner"])
print("Rótulos POS (amostra):", meta["labels"]["morphologizer"][:6])

print("\nDesempenho declarado:")
for k, v in meta["performance"].items():
    if isinstance(v, float):
        print(f"  {k:20s} {v:.3f}")

Licença do modelo: CC BY-SA 4.0
Vetores: {'width': 0, 'vectors': 0, 'keys': 0, 'name': None, 'mode': 'default'}

Corpora de origem (sources) usados no treinamento:
  • UD Portuguese Bosque v2.8
      licença: CC BY-SA 4.0
      url    : https://github.com/UniversalDependencies/UD_Portuguese-Bosque
  • WikiNER
      licença: CC BY 4.0
      url    : https://figshare.com/articles/Learning_multilingual_named_entity_recognition_from_Wikipedia/5462500

Rótulos de entidade nomeada: ['LOC', 'MISC', 'ORG', 'PER']
Rótulos POS (amostra): ['Definite=Ind|Gender=Masc|Number=Sing|POS=DET|PronType=Art', 'Gender=Masc|Number=Sing|POS=NOUN', 'Gender=Masc|Number=Sing|POS=ADJ', 'Definite=Def|Gender=Masc|Number=Sing|POS=DET|PronType=Art', 'Gender=Masc|Number=Sing|POS=PROPN', 'Number=Sing|POS=PROPN']

Desempenho declarado:
  sents_p              0.926
  sents_r              0.946
  sents_f              0.922
  tag_acc              0.889
  ents_p               0.881
  ents_r               0.882
  ents_f     

### 1.5 Dicionário de rótulos linguísticos (`spacy.explain`)

In [9]:
for rot in ["NOUN", "VERB", "ADJ", "DET", "ADP", "PROPN", "PER", "LOC", "ORG", "nsubj", "obj"]:
    print(f"  {rot:8s} -> {spacy.explain(rot)}")

  NOUN     -> noun
  VERB     -> verb
  ADJ      -> adjective
  DET      -> determiner
  ADP      -> adposition
  PROPN    -> proper noun
  PER      -> Named person or family.
  LOC      -> Non-GPE locations, mountain ranges, bodies of water
  ORG      -> Companies, agencies, institutions, etc.
  nsubj    -> nominal subject
  obj      -> object


### 1.6 Vocabulário / léxico do modelo

O `Vocab` guarda lexemas com atributos calculados por regra (não dependem de contexto).

In [10]:
print("Lexemas em cache no vocabulário:", len(nlp.vocab))

for palavra in ["chamado", "R$", "12/09/2026", "NOC", "servidor123"]:
    lex = nlp.vocab[palavra]
    print(f"  {palavra:12s} alpha={lex.is_alpha!s:5s} digito={lex.is_digit!s:5s} "
          f"stop={lex.is_stop!s:5s} shape={lex.shape_}")

Lexemas em cache no vocabulário: 355
  chamado      alpha=True  digito=False stop=False shape=xxxx
  R$           alpha=False digito=False stop=False shape=X$
  12/09/2026   alpha=False digito=False stop=False shape=dd/dd/dddd
  NOC          alpha=True  digito=False stop=False shape=XXX
  servidor123  alpha=False digito=False stop=False shape=xxxxddd


### 1.7 Pipeline completo aplicado a uma frase embutida

Junta tudo: tokenização → lematização → POS → dependência → NER.

In [11]:
doc_demo = nlp(frases_pt[0])

tabela_demo = pd.DataFrame(
    [(t.text, t.lemma_, t.pos_, t.tag_, t.dep_, t.is_stop, t.is_punct) for t in doc_demo],
    columns=["Token", "Lema", "POS", "TAG", "DEP", "is_stop", "is_punct"],
)
display(tabela_demo)

print("Frase:", doc_demo.text)
print("Entidades nomeadas:", [(e.text, e.label_, spacy.explain(e.label_)) for e in doc_demo.ents])

,Token,Lema,POS,TAG,DEP,is_stop,is_punct
0,Apple,Apple,PROPN,PROPN,nsubj,False,False
1,está,estar,AUX,AUX,aux,True,False
2,querendo,querer,VERB,VERB,ROOT,False,False
3,comprar,comprar,VERB,VERB,xcomp,False,False
4,uma,um,DET,DET,det,True,False
5,startup,startup,NOUN,NOUN,obj,False,False
6,do,de o,ADP,ADP,case,True,False
7,Reino,Reino,PROPN,PROPN,nmod,False,False
8,Unido,Unido,PROPN,PROPN,flat:name,False,False
9,por,por,ADP,ADP,case,True,False


Frase: Apple está querendo comprar uma startup do Reino Unido por 100 milhões de dólares
Entidades nomeadas: [('Apple', 'ORG', 'Companies, agencies, institutions, etc.'), ('Reino Unido', 'LOC', 'Non-GPE locations, mountain ranges, bodies of water')]


---
## Parte 2 — Acesso ao corpora SUBTLEX-pt-BR

**Ficha do corpus**

| Item | Valor |
|---|---|
| Nome | SUBTLEX-PT-BR |
| Autor | Kevin Tang (UCL), 2012 |
| Origem dos dados | legendas de filmes/séries em pt-BR do **OpenSubtitles.org** |
| Tamanho | **61.609.241** ocorrências (tokens) / **136.147** tipos (types) |
| Documentos | 12.104 arquivos de legenda distintos |
| Distribuição | OSF — https://osf.io/vb5yp/ (arquivos `.tsv`) |
| Licença | CC BY-NC-ND 4.0 (uso não comercial, sem obras derivadas) |
| Referência | TANG, K. *A 61 Million Word Corpus of Brazilian Portuguese Film Subtitles as a Resource for Linguistic Research.* UCL Working Papers in Linguistics, v. 24, p. 208–214, 2012. |

**Importante:** por restrição de direitos autorais, o corpus **não distribui os textos das legendas**.
O que se distribui é a tabela de frequências agregada — ou seja, um corpus já reduzido a
*características* (etapa 2 da pipeline de PLN), não a texto bruto.

**Colunas do arquivo (os metadados do corpora):**

| Coluna | Significado |
|---|---|
| `Word` | forma da palavra, em minúsculas, apenas caracteres do alfabeto português |
| `FREQcount` | número total de ocorrências da palavra no corpus inteiro |
| `CDcount` | *contextual diversity* — em quantos dos 12.104 arquivos de legenda a palavra aparece |
| `Spellcheck` | `TRUE`/`FALSE` — se a forma passa na verificação ortográfica de um dicionário pt-BR |

Palavras com `CDcount <= 2` foram removidas na construção do corpus (filtro contra ruído de
legendas mal transcritas).

### 2.1 Download com cache local

In [12]:
URLS_SUBTLEX = [
    # Arquivo completo: 136.147 tipos, incluindo entradas com Spellcheck = FALSE
    "https://osf.io/download/p43sy/",
    "https://files.osf.io/v1/resources/vb5yp/providers/osfstorage/6806ce5f37a194040a35fe82?direct",
]
ARQ_LOCAL = "SUBTLEX_PT-BR_CDAbove2_Alpha_Spellcheck.tsv"


def baixar_subtlex(destino=ARQ_LOCAL, urls=URLS_SUBTLEX):
    '''Baixa o SUBTLEX-pt-BR do OSF (osf.io/vb5yp) e mantém cache local.'''
    if os.path.exists(destino):
        print(f"Já existe em cache: {destino} ({os.path.getsize(destino)/1e6:.2f} MB)")
        return destino
    ultimo_erro = None
    for url in urls:
        try:
            print(f"Baixando de {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=180) as r, open(destino, "wb") as f:
                f.write(r.read())
            print(f"OK -> {destino} ({os.path.getsize(destino)/1e6:.2f} MB)")
            return destino
        except Exception as e:
            ultimo_erro = e
            print("   falhou:", e)
    raise RuntimeError(
        "Download automático falhou (rede/proxy). Baixe manualmente em https://osf.io/vb5yp/ "
        f"o arquivo 'SUBTLEX_PT-BR_CDAbove2_Alpha_Spellcheck.tsv' e salve nesta pasta. "
        f"Último erro: {ultimo_erro}"
    )


baixar_subtlex()

Baixando de https://osf.io/download/p43sy/ ...
OK -> SUBTLEX_PT-BR_CDAbove2_Alpha_Spellcheck.tsv (2.73 MB)


'SUBTLEX_PT-BR_CDAbove2_Alpha_Spellcheck.tsv'

### 2.2 Carregamento e inspeção dos dados

In [13]:
subtlex = pd.read_csv(ARQ_LOCAL, sep="\t", encoding="utf-8",
                      keep_default_na=False, na_values=[])
subtlex["Word"] = subtlex["Word"].astype(str)

# a coluna pode ser lida como bool ou como texto "TRUE"/"FALSE": normalizamos os dois casos
subtlex["Spellcheck"] = subtlex["Spellcheck"].astype(str).str.strip().str.upper().eq("TRUE")
subtlex = subtlex.drop_duplicates(subset="Word", keep="first").reset_index(drop=True)

print("Formato (linhas, colunas):", subtlex.shape)
print("Tipos de dado:\n", subtlex.dtypes, sep="")
display(subtlex.head(10))

Formato (linhas, colunas): (136147, 4)
Tipos de dado:
Word          object
FREQcount      int64
CDcount        int64
Spellcheck      bool
dtype: object


,Word,FREQcount,CDcount,Spellcheck
0,a,1349974,12092,True
1,aa,210,132,False
2,aaa,46,23,False
3,aaaa,12,9,False
4,aaaaa,3,3,False
5,aaaaaaaaaah,3,3,False
6,aaaaaaah,4,4,False
7,aaaaaah,7,7,False
8,aaaaah,21,14,False
9,aaaaahhhh,4,4,False


### 2.3 Metadados agregados e medidas derivadas

A partir das três colunas brutas derivam-se as medidas usadas na literatura psicolinguística:

- **SUBTLWF** = frequência por milhão de palavras — permite comparar corpora de tamanhos diferentes.
- **Lg10WF** = log₁₀(frequência+1) — a distribuição de frequências é fortemente assimétrica (lei de Zipf).
- **CDpct** = percentual de documentos em que a palavra aparece.
- **Zipf** = log₁₀(frequência por bilhão) — escala de 1 (muito rara) a 7 (muito comum), de leitura direta.

In [14]:
TOTAL_TOKENS = int(subtlex["FREQcount"].sum())
TOTAL_DOCS = int(subtlex["CDcount"].max())

subtlex["SUBTLWF"] = subtlex["FREQcount"] / TOTAL_TOKENS * 1_000_000
subtlex["Lg10WF"] = (subtlex["FREQcount"] + 1).apply(math.log10)
subtlex["CDpct"] = subtlex["CDcount"] / TOTAL_DOCS * 100
subtlex["Zipf"] = (subtlex["FREQcount"] / TOTAL_TOKENS * 1e9).apply(math.log10)

print("=== Metadados agregados do SUBTLEX-pt-BR ===")
print(f"  tipos (types)                : {len(subtlex):,}")
print(f"  ocorrências (tokens)         : {TOTAL_TOKENS:,}")
print(f"  arquivos de legenda (max CD) : {TOTAL_DOCS:,}")
print(f"  Spellcheck = TRUE            : {int(subtlex['Spellcheck'].sum()):,}")
print(f"  Spellcheck = FALSE           : {int((~subtlex['Spellcheck']).sum()):,}")
print(f"  razão type/token             : {len(subtlex)/TOTAL_TOKENS:.6f}")

=== Metadados agregados do SUBTLEX-pt-BR ===
  tipos (types)                : 136,147
  ocorrências (tokens)         : 61,609,241
  arquivos de legenda (max CD) : 12,096
  Spellcheck = TRUE            : 78,908
  Spellcheck = FALSE           : 57,239
  razão type/token             : 0.002210


### 2.4 Uso típico dos **dados**: as palavras mais frequentes da língua falada

In [15]:
display(
    subtlex.nlargest(15, "FREQcount")[
        ["Word", "FREQcount", "CDcount", "Spellcheck", "SUBTLWF", "CDpct", "Zipf"]
    ].round(2)
)

,Word,FREQcount,CDcount,Spellcheck,SUBTLWF,CDpct,Zipf
105018,que,2135010,12089,True,34654.05,99.94,7.54
91030,o,1773371,12096,True,28784.17,100.00,7.46
90810,não,1764384,12023,True,28638.30,99.40,7.46
35099,de,1447464,12095,True,23494.27,99.99,7.37
0,a,1349974,12092,True,21911.88,99.97,7.34
135830,é,1204763,11746,True,19554.91,97.11,7.29
44795,e,1057272,12089,True,17160.93,99.94,7.23
132922,você,1010350,11323,True,16399.33,93.61,7.21
53741,eu,976825,11965,True,15855.17,98.92,7.20
129356,um,823472,12077,True,13366.05,99.84,7.13


### 2.5 Uso típico dos **metadados**

Dois usos que só existem por causa dos metadados:

**(a) `Spellcheck`** separa vocabulário real do ruído típico de legenda — onomatopeias, nomes
próprios, palavras estrangeiras, erros de digitação e contrações sem hífen/apóstrofo (`deixeme`,
`digame`, `fazêlo`). Para montar um léxico de referência, filtra-se por `Spellcheck == True`.

**(b) `CDcount` vs `FREQcount`** distingue dois perfis de palavra:
- **frequência alta + CD alto** → palavra corrente da língua (candidata a *stop word*);
- **frequência alta + CD baixo** → palavra concentrada em poucos documentos, normalmente nome de
  personagem ou termo temático — exatamente o que **não** se quer numa lista de stop words.

É a mesma distinção que motiva o TF-IDF: frequência sozinha engana.

In [16]:
print(">>> (a) Entradas frequentes reprovadas na verificação ortográfica:")
display(subtlex[~subtlex["Spellcheck"]].nlargest(10, "FREQcount")[["Word", "FREQcount", "CDcount"]])

print(">>> (b) Frequência ALTA e diversidade contextual BAIXA (termos concentrados):")
concentradas = subtlex[(subtlex["FREQcount"] > 3000) & (subtlex["CDpct"] < 12)]
display(concentradas.nlargest(10, "FREQcount")[["Word", "FREQcount", "CDcount", "CDpct", "Spellcheck"]].round(2))

print(">>> (b) Frequência ALTA e diversidade contextual ALTA (língua corrente):")
correntes = subtlex[(subtlex["CDpct"] > 95) & (subtlex["Spellcheck"])]
display(correntes.nlargest(10, "FREQcount")[["Word", "FREQcount", "CDcount", "CDpct"]].round(2))

>>> (a) Entradas frequentes reprovadas na verificação ortográfica:


,Word,FREQcount,CDcount
92143,ok,37152,4554
67493,idéia,17541,6100
120142,sra,16916,3745
36196,deixeme,16227,6492
53567,estã,13998,347
132908,voce,12242,621
73975,john,11871,2169
120155,srta,10994,2163
73168,jack,10255,1412
116457,sentese,10088,4987


>>> (b) Frequência ALTA e diversidade contextual BAIXA (termos concentrados):


,Word,FREQcount,CDcount,CDpct,Spellcheck
135747,ã,25841,373,3.08,True
53567,estã,13998,347,2.87,False
67210,i,12329,1398,11.56,True
132908,voce,12242,621,5.13,False
53543,estás,10534,1355,11.20,True
73168,jack,10255,1412,11.67,False
114530,sam,8132,1061,8.77,False
73848,joe,7454,1148,9.49,False
24037,charlie,7288,1109,9.17,False
124192,tens,7158,1192,9.85,True


>>> (b) Frequência ALTA e diversidade contextual ALTA (língua corrente):


,Word,FREQcount,CDcount,CDpct
105018,que,2135010,12089,99.94
91030,o,1773371,12096,100.00
90810,não,1764384,12023,99.40
35099,de,1447464,12095,99.99
0,a,1349974,12092,99.97
135830,é,1204763,11746,97.11
44795,e,1057272,12089,99.94
53741,eu,976825,11965,98.92
129356,um,823472,12077,99.84
94553,para,672331,12051,99.63


---
## Parte 3 — Aplicação a um problema simples de PLN

### O problema

Uma central de serviços (*helpdesk*) recebe chamados escritos em texto livre. Antes de qualquer
classificação automática, é preciso **pré-processar o texto e sinalizar termos problemáticos**:

- palavras **fora do vocabulário corrente** → jargão técnico ou erro de digitação, que quebram
  modelos treinados em linguagem geral;
- estimar a **dificuldade lexical** do chamado, para roteirizar o atendimento.

### A solução

Combinar as duas fontes deste notebook:

| Fonte | Papel |
|---|---|
| **spaCy** (`pt_core_news_sm`) | tokeniza, lematiza e classifica gramaticalmente (POS) |
| **SUBTLEX-pt-BR** | diz, para cada forma, **quão comum ela é na língua falada** e se é dicionarizada |

A lematização do spaCy é essencial: o SUBTLEX indexa *formas*, então `travou` pode não estar
listada — mas o lema `travar` está. A consulta é feita em cascata: forma → lema.

### 3.1 Índice de consulta e regra de classificação

In [17]:
FREQ = subtlex.set_index("Word")[["FREQcount", "CDcount", "Spellcheck", "Zipf", "CDpct"]].to_dict("index")


def consultar(palavra):
    '''Consulta uma forma no SUBTLEX-pt-BR (retorna None se ausente).'''
    return FREQ.get(palavra.lower())


def classificar(info):
    '''Regra simples baseada nos metadados do corpora.'''
    if info is None:
        return "FORA-DO-CORPUS"      # jargão, neologismo ou erro de digitação
    if not info["Spellcheck"]:
        return "NAO-DICIONARIZADA"   # forma reprovada no dicionário pt-BR
    if info["Zipf"] >= 5:
        return "MUITO-COMUM"
    if info["Zipf"] >= 3:
        return "COMUM"
    return "RARA"


def analisar(texto):
    '''spaCy + SUBTLEX: uma linha por token, com dados e metadados do corpora.'''
    doc = nlp(texto)
    linhas = []
    for t in doc:
        if t.is_punct or t.is_space:
            continue
        info = consultar(t.text) or consultar(t.lemma_)
        linhas.append({
            "Token": t.text,
            "Lema": t.lemma_,
            "POS": t.pos_,
            "stop_spaCy": t.is_stop,
            "FREQcount": info["FREQcount"] if info else None,
            "CDcount": info["CDcount"] if info else None,
            "Spellcheck": info["Spellcheck"] if info else None,
            "Zipf": round(info["Zipf"], 2) if info else None,
            "Classe": classificar(info),
        })
    return pd.DataFrame(linhas)

### 3.2 Os chamados de entrada

In [18]:
CHAMADOS = [
    "Bom dia, meu computador não está ligando desde ontem.",
    "O cliente relatou que o notebook não liga depois da atualização do firmware.",
    "A anastomose foi realizada sem intercorrências durante o procedimento cirúrgico.",
    "Meu compputador travou e nao consigo abrir o sisttema de chamados.",
]

for i, c in enumerate(CHAMADOS, 1):
    print(f"{i}. {c}")

1. Bom dia, meu computador não está ligando desde ontem.
2. O cliente relatou que o notebook não liga depois da atualização do firmware.
3. A anastomose foi realizada sem intercorrências durante o procedimento cirúrgico.
4. Meu compputador travou e nao consigo abrir o sisttema de chamados.


### 3.3 Análise token a token (chamado 4, com erros de digitação)

In [19]:
display(analisar(CHAMADOS[3]))

,Token,Lema,POS,stop_spaCy,FREQcount,CDcount,Spellcheck,Zipf,Classe
0,Meu,meu,DET,True,241367.0,11739.0,True,6.59,MUITO-COMUM
1,compputador,compputador,NOUN,False,NaN,NaN,None,NaN,FORA-DO-CORPUS
2,travou,travar,VERB,False,92.0,83.0,True,3.17,COMUM
3,e,e,CCONJ,True,1057272.0,12089.0,True,7.23,MUITO-COMUM
4,nao,nao,PROPN,False,5918.0,439.0,False,4.98,NAO-DICIONARIZADA
5,consigo,consigo,NOUN,False,15977.0,6801.0,True,5.41,MUITO-COMUM
6,abrir,abrir,VERB,False,6483.0,4130.0,True,5.02,MUITO-COMUM
7,o,o,DET,True,1773371.0,12096.0,True,7.46,MUITO-COMUM
8,sisttema,sisttema,NOUN,False,NaN,NaN,None,NaN,FORA-DO-CORPUS
9,de,de,ADP,True,1447464.0,12095.0,True,7.37,MUITO-COMUM


### 3.4 Índice de dificuldade lexical

Média do Zipf das palavras de conteúdo (substantivo, verbo, adjetivo, advérbio, nome próprio).
**Zipf baixo = vocabulário raro = texto lexicalmente difícil.**

In [20]:
def dificuldade_lexical(texto):
    df = analisar(texto)
    conteudo = df[df["POS"].isin(["NOUN", "VERB", "ADJ", "ADV", "PROPN"])]
    zipfs = conteudo["Zipf"].dropna()
    return (round(zipfs.mean(), 2) if len(zipfs) else float("nan"),
            int(conteudo["Zipf"].isna().sum()),
            len(conteudo))


linhas = []
for i, c in enumerate(CHAMADOS, 1):
    z, oov, n = dificuldade_lexical(c)
    linhas.append({"#": i, "Zipf_medio": z, "fora_do_corpus": oov,
                   "palavras_conteudo": n, "Chamado": c})

ranking = pd.DataFrame(linhas).sort_values("Zipf_medio").reset_index(drop=True)
display(ranking)
print("Do mais difícil (Zipf menor) para o mais fácil (Zipf maior).")

,#,Zipf_medio,fora_do_corpus,palavras_conteudo,Chamado
0,3,3.21,1,5,A anastomose foi realizada sem intercorrências...
1,4,4.54,2,7,Meu compputador travou e nao consigo abrir o s...
2,2,4.60,1,8,O cliente relatou que o notebook não liga depo...
3,1,5.70,0,6,"Bom dia, meu computador não está ligando desde..."


Do mais difícil (Zipf menor) para o mais fácil (Zipf maior).


### 3.5 Saída do sistema: termos sinalizados para revisão

In [21]:
for i, c in enumerate(CHAMADOS, 1):
    df = analisar(c)
    flag = df[df["Classe"].isin(["FORA-DO-CORPUS", "NAO-DICIONARIZADA", "RARA"])]
    print(f"\nChamado {i}: {c}")
    if len(flag) == 0:
        print("   nenhum termo sinalizado — vocabulário totalmente corrente")
    else:
        for _, r in flag.iterrows():
            print(f"   [{r['Classe']:18s}] {r['Token']:16s} (lema: {r['Lema']}, Zipf: {r['Zipf']})")


Chamado 1: Bom dia, meu computador não está ligando desde ontem.
   nenhum termo sinalizado — vocabulário totalmente corrente

Chamado 2: O cliente relatou que o notebook não liga depois da atualização do firmware.
   [RARA              ] notebook         (lema: notebook, Zipf: 2.85)
   [FORA-DO-CORPUS    ] firmware         (lema: firmware, Zipf: nan)

Chamado 3: A anastomose foi realizada sem intercorrências durante o procedimento cirúrgico.
   [RARA              ] anastomose       (lema: anastomose, Zipf: 2.11)
   [FORA-DO-CORPUS    ] intercorrências  (lema: intercorrência, Zipf: nan)

Chamado 4: Meu compputador travou e nao consigo abrir o sisttema de chamados.
   [FORA-DO-CORPUS    ] compputador      (lema: compputador, Zipf: nan)
   [NAO-DICIONARIZADA ] nao              (lema: nao, Zipf: 4.98)
   [FORA-DO-CORPUS    ] sisttema         (lema: sisttema, Zipf: nan)


### 3.6 Segunda aplicação: lista de stop words orientada a dados

Em vez de usar a lista fixa do spaCy, deriva-se uma lista **a partir do corpus**, usando os dois
metadados juntos: alta frequência **e** alta diversidade contextual. Depois compara-se com a lista
embutida do spaCy — é a diferença prática entre uma abordagem baseada em regras/especialistas e uma
abordagem baseada em dados (tema do exercício TC.28).

In [22]:
candidatas = subtlex[(subtlex["CDpct"] >= 90) & (subtlex["Spellcheck"])].nlargest(80, "FREQcount")
set_dados = set(candidatas["Word"])
set_spacy = {w.lower() for w in STOP_PT}

print(f"Candidatas derivadas do corpus (top 80, CD >= 90%): {len(set_dados)}")
print(f"  também presentes na lista do spaCy : {len(set_dados & set_spacy)}")
print(f"  ausentes da lista do spaCy         : {len(set_dados - set_spacy)}")
print("\nPalavras que o corpus indica como stop word mas o spaCy não lista:")
print("  ", sorted(set_dados - set_spacy))

Candidatas derivadas do corpus (top 80, CD >= 90%): 80
  também presentes na lista do spaCy : 75
  ausentes da lista do spaCy         : 5

Palavras que o corpus indica como stop word mas o spaCy não lista:
   ['disse', 'há', 'mim', 'vamos', 'vou']


---
## Conclusões

1. **spaCy e SUBTLEX-pt-BR são recursos complementares, não concorrentes.** O spaCy fornece
   *processamento* (tokenização, lema, POS, NER) e traz consigo dados linguísticos auxiliares
   (stop words, exceções, frases de exemplo) e metadados que apontam para seus corpora de treino
   (UD Portuguese Bosque e WikiNER). O SUBTLEX-pt-BR fornece *evidência estatística de uso real*
   da língua falada — algo que o spaCy não tem.

2. **Os metadados carregam a informação decisiva.** `FREQcount` sozinho engana: só com `CDcount`
   se distingue palavra corrente de palavra concentrada em poucos documentos, e só com `Spellcheck`
   se separa vocabulário legítimo de ruído de legenda. É o mesmo princípio que justifica o TF-IDF.

3. **O registro do corpus define sua aplicabilidade.** Por vir de legendas, o SUBTLEX-pt-BR
   aproxima-se da língua falada — Tang (2012) mostrou correlação maior com corpus conversacional
   (r = 0,54) do que com corpus escrito (r = 0,50). Isso o torna adequado a chatbots, triagem e
   texto de usuário, e inadequado como referência para texto técnico ou jurídico: `firmware` estar
   fora do corpus não significa que não seja palavra — significa que não é palavra *de filme*.

4. **Limitações práticas.** O corpus é de 2012 (não contém vocabulário digital recente), é uma
   lista de unigramas sem POS nem lema (daí a necessidade de lematizar com o spaCy antes de
   consultar) e está sob licença **CC BY-NC-ND 4.0** — uso não comercial e sem obras derivadas,
   restrição relevante para qualquer sistema de PLN em produção.